# GeoSpatial Visualization

In [ ]:
# System Path #
import os
import sys 

# Add dsci_550_a1 to base path. Lets you project functions #
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

# Pandas #
import pandas as pd
import time
import re

# Runtime #
import time
from tqdm import tqdm 

# Iterators #
import collections
import ast
import random

# Flight Trajectory Functions #
from dsci_550_a1.flightFunctions import *

# Plotting #
import plotly.graph_objects as go


## Load Data

In [ ]:
df_haunted_places = pd.read_csv("../data/processed/haunted_places_features_added.tab", sep = "\t")
df_american_routes = pd.read_csv("../data/joined_datasets/american_routes.tsv", sep = "\t")
df_american_airports = pd.read_csv("../data/joined_datasets/american_airports.tsv", sep = "\t")

## Add Haunting Types ## 
df_haunted_places['Haunting_Type'] = pd.Series(np.array(['UFO'] * 5000 + ['Ghost'] * 3000 + ['Mansion'] * 2991))


## Pandas stores nested dicts and lists as strings ##
## This converts them back to lists and dicst ##

df_american_routes["Flight_Path"] = df_american_routes["Flight_Path"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df_american_airports["Airport_Radius"] = df_american_airports["Airport_Radius"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df_haunted_places["Flight_Intersections"] = df_haunted_places["Flight_Intersections"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df_haunted_places["Aerodrome_Intersections"] = df_haunted_places["Aerodrome_Intersections"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

## Filter Data

In [ ]:
(df_american_airports["Iata_Code"] == "LAX").idxmax()

In [ ]:
## Choose indices of interest in haunted_places dataframe ##
m = df_haunted_places.shape[0]
haunted_places_indices = [random.randrange(0,m) for x in range(100)]
haunted_places_indices = df_haunted_places[df_haunted_places["Flight_HighTraffic"] == True].index.to_list()
route_indices = []
airport_indices =[]

## Add all airports and routes that correspond to filtered haunted places ##
for row in df_haunted_places.loc[haunted_places_indices].itertuples(index = False):

    # Add all route indices that correspond to haunted place of interest
    [route_indices.append(flight["Route_ID"]) for flight in row.Flight_Intersections if flight["Route_ID"] not in route_indices]  

    # Add source and destination airport indices if they aren't already in the list
    [airport_indices.append((df_american_airports["Iata_Code"] == flight["Source_Airport"]).idxmax()) for flight in row.Flight_Intersections if (df_american_airports["Iata_Code"] == flight["Source_Airport"]).idxmax() not in route_indices]
    [airport_indices.append((df_american_airports["Iata_Code"] == flight["Dest_Airport"]).idxmax()) for flight in row.Flight_Intersections if (df_american_airports["Iata_Code"] == flight["Dest_Airport"]).idxmax() not in route_indices]

    # Add all airport indices that are close to haunted place of interest
    [airport_indices.append(airport["Airport_ID"]) for airport in row.Aerodrome_Intersections if airport["Airport_ID"] not in route_indices]

## Filter routes
routes_filtered = df_american_routes.loc[route_indices, ["Flight_Path"]]

## Filter Airports
airports_filtered = df_american_airports.loc[airport_indices]

## Filter Haunted Places
haunted_places_filtered = df_haunted_places.loc[haunted_places_indices]

In [ ]:
airports_filtered

In [ ]:
## Create Plotly figure ##
fig = go.Figure()

## Add Haunting Types ##
haunt_types = haunted_places_filtered['Haunting_Type'].unique().tolist()
haunting_plot_colors = ['rgb(127,255,0)', 'rgb(218,112,214)', 'rgb(138,43,226)']


for i, haunting_type in enumerate(haunt_types):
    haunts_to_plot = haunted_places_filtered.loc[haunted_places_filtered['Haunting_Type'] == haunting_type]
    trace = (go.Scattergeo(
        locationmode = 'USA-states',
        lon = haunts_to_plot['longitude'],
        lat = haunts_to_plot['latitude'],
        hoverinfo = 'text',
        text = haunts_to_plot['Haunting_Type'],
        mode = 'markers',
        showlegend = False, 
        marker = dict(
            size = 4,
            color = haunting_plot_colors[i],
            opacity = 0.5
            ),
            name = haunting_type,
            visible = False
        )
    )
    fig.add_trace(trace)
    

## Unpack Coords ##
lats_plot, lons_plot = [] , []
for row in routes_filtered.itertuples(index = False):   

    ## Unpack Coords ##
    lats, lons = zip(*row.Flight_Path)
    lats, lons = list(lats), list(lons)

    lats_plot.extend(lats + [None])
    lons_plot.extend(lons + [None])

## Add Flight Paths ##
fig.add_trace(go.Scattergeo(
    lon= lons_plot,
    lat= lats_plot,
    mode='lines',
    line=dict(width=.5, color='red'),
    opacity = 0.2, 
    hoverinfo = 'skip', 
    name = "Flights",
    visible = False
))

## Add Airports ##

airport_types = airports_filtered['Type'].unique().tolist()

airport_plot_colors = {
'heliport' :        "rgb(100,151,177)" ,
 'seaplane_base': 	"rgb(179,205,224)",
 'balloonport' : 	"rgb(179,205,224)",
 'small_airport' :  "rgb(0,91,150)"  ,
 'medium_airport' :	"rgb(3,57,108)",
 'large_airport':   "rgb(1,31,75)"
}

airport_proximity_dict = {
    "large_airport" : 55560,    # 30 nautical miles
    "medium_airport" : 9260,    # 5 nautical miles
    "small_airport" : 5556,     # 3 nautical miles
    "heliport":  2778,          # 1.5 nautical miles
    "seaplane_base" : 5556,     # 3 nautical miles
    "balloonport" : 5556        # 3 nautical miles
}

## Plot airports ##
for airport_type in airport_types:

    ## Airport Marker Trace##
    airports_to_plot = airports_filtered.loc[airports_filtered['Type'] == airport_type]

    airports_trace = (go.Scattergeo(
    locationmode = 'USA-states',
    lon = airports_to_plot['Longitude_Deg'],
    lat = airports_to_plot['Latitude_Deg'],
    hoverinfo = 'text',
    text = airports_to_plot['Iata_Code'],
    mode = 'markers',
    marker = dict(
        size = 1,
        color = airport_plot_colors[airport_type],
        opacity = 1
        ),
        name = airport_type,
        visible = False
        ))
    
    ## Airport Radius Trace ##
    lons_plot = [] 
    lats_plot = []
    for airport in airports_to_plot.itertuples():
        ## Unpack Coords ##
        lats, lons = zip(*airport.Airport_Radius)
        lats, lons = list(lats), list(lons)

        lats_plot.extend(lats + [None])
        lons_plot.extend(lons + [None])
    
    airport_radii = (go.Scattergeo(
    locationmode = 'USA-states',
    lon = lons_plot,
    lat = lats_plot,
    hoverinfo = 'skip',
    mode = 'lines',
    line = dict(
        width = 1,
        color = airport_plot_colors[airport_type],
        dash = 'dot'
        ),
        name = airport_type,
        visible = False
        ))
    
    fig.add_trace(airports_trace)
    fig.add_trace(airport_radii)
## Plot Radius of Airports ##


## Add Interactive Buttons for Haunts ##
buttons_haunts = [
            {
                "label"  : haunt_type,
                "method" : "update",
                "args"   :  [{"visible" : [haunt_type == h for h in haunt_types] + [False] + [False] * len(airport_types) * 2}],
            }
 for haunt_type in haunt_types
]


buttons_haunts.append(
            {
                "label"  : "Show Haunts and Flight Paths",
                "method" : "update",
                "args"   :  [{"visible" : [True] * len(haunt_types) + [True] + [False] * len(airport_types) * 2}],
            },
        )

buttons_haunts.append(
            {
                "label"  : "Show Haunts and Airports",
                "method" : "update",
                "args"   :  [{"visible" : [True] * len(haunt_types) + [False] + [True] * len(airport_types) * 2}],
            },)

## Add Interactive Buttons for Flights ##
buttons_flights = [
            {
                "label"  : "Show Flights",
                "method" : "update",
                "args"   :  [{"visible" : [[False] * len(haunt_types) + [True] + [False] * len(airport_types) * 2]}]
            },
                        {
                "label"  : "Show Airports",
                "method" : "update",
                "args"   :  [{"visible" : [[False] * len(haunt_types) + [False] + [True] * len(airport_types) * 2]}]
            },
                        {
                "label"  : "Show Flights and Airports",
                "method" : "update",
                "args"   :  [{"visible" : [{"visible" : [[True] * len(haunt_types) + [True] + [True] * len(airport_types) * 2]}]}]
            }]


fig.update_layout(
    title_text = 'Flight Paths Accross U.S.',
    showlegend = True,
    geo = dict(
        scope = 'north america',
        projection_type = 'azimuthal equal area',
        showland = True,
        showcountries = True,
        showsubunits = True, 
        subunitcolor = "Black",
        landcolor = 'rgb(243, 243, 243)',
        countrycolor = 'rgb(204, 204, 204)',
    ),
    updatemenus=[
    {
        "buttons": buttons_haunts,
        "direction" : "down",
        "showactive" : True,
        "x": 0.1,
        "y": 1.15,
        "xanchor" : "left",
        "yanchor" : "top", 
        "font": {"size" : 12},
        "type": "dropdown",
        "name": "Haunt Toggle"
    },
    {
        "buttons": buttons_flights,
        "direction" : "down",
        "showactive" : True,
        "x": 0.1,
        "y": 1.0,
        "xanchor" : "left",
        "yanchor" : "top", 
        "font": {"size" : 12},
        "type": "dropdown",
        "name": "Flight Toggle"
    }
    ]
)

fig.write_html("flight_paths.html")


# Show plot
fig.show()

### All Airline Routes

In [ ]:

import random

m = df_haunted_places.shape[0]

haunted_places_indices = [random.randrange(0,m) for x in range(100)]
route_indices = []
airport_indices =[]
for row in df_haunted_places.loc[haunted_places_indices].itertuples(index = False):
    [route_indices.append(flight["Route_ID"]) for flight in row.Flight_Intersections if flight["Route_ID"] not in route_indices]
    [airport_indices.append(airport["Airport_ID"]) for airport in row.Aerodrome_Intersections if airport["Airport_ID"] not in route_indices]

# routes_indices = df_haunted_places.loc[haunted_places_indices, "Flight_Intersections"].apply(lambda x: x[1][''])
routes_filtered = df_american_routes.loc[route_indices, "Flight_Path"]
airports_filtered = df_american_airports.loc[airport_indices]


In [ ]:
df_haunted_places["Flight_Intersections"][0][0]["Route_ID"]

In [ ]:


# paths = [x.Flight_Path for x in df_american_routes.itertuples(index=False)]

# x = [(33.942501, -118.40799700000001), (35.3391826757794, -114.40879856305514), (36.59969697040937, -110.2759806905069), (37.71299048499094, -106.01411235472213), (38.66844452370027, -101.63167373007136), (39.45623426747724, -97.14131816943458), (40.0677169768164, -92.55987920972437), (40.49581951847608, -87.90805848416996), (40.73538850700927, -83.20976646875951), (40.783464526482796, -78.49113990558394), (40.639447, -73.779317)]

paths = routes_filtered
ts = np.ceil(np.linspace(1, 10000, num = 2)).astype(int)

## Unpack all flight paths ##
# paths = [x.Flight_Path for x in df_american_routes.itertuples(index=False)]

## Calculate Path Radius ##
path_radii = []

res = []

# for t in ts:
#     xpos, xneg = flight_trajectory_radius(x, d = t)
#     paths.append(xpos), paths.append(xneg)

for path in paths[:]:
    for t in ts[1:]:
        xpos, xneg = flight_trajectory_radius(path, d = t)
        path_radii.extend([xpos,xneg])

lats_plot = []
lons_plot = [] 

for path in paths[:]:
    lats, lons = zip(*path)
    lats, lons = list(lats), list(lons)

    lats_plot.extend(lats + [None])
    lons_plot.extend(lons + [None])


flight_traces = [
    go.Scattergeo(
        lon=lons_plot,  
        lat=lats_plot,  
        mode="lines",
        line=dict(color="red", width=1),
        showlegend=False,
        hoverinfo = 'skip', 
        visible = True
    )
]

lats_plot = []
lons_plot = [] 

for radii in path_radii:
    lats, lons = zip(*radii)
    lats, lons = list(lats), list(lons)

    lats_plot.extend(lats + [None])
    lons_plot.extend(lons + [None])


radii_traces = [
    go.Scattergeo(
        lon=lons_plot,  
        lat=lats_plot,  
        mode="lines",
        line=dict(color="blue", width=.5, dash = 'dot'),
        showlegend=False,
        hoverinfo = 'skip', 
        visible = True
    )
]  


# Combine all traces
fig = go.Figure(data=flight_traces + radii_traces)


# Set map layout
fig.update_layout(
    title_text = 'Flight Paths Accross U.S.',
    showlegend = True,
    geo = dict(
        scope = 'north america',
        projection_type = 'azimuthal equal area',
        showland = True,
        showcountries = True,
        showsubunits = True, 
        subunitcolor = "Black",
        landcolor = 'rgb(243, 243, 243)',
        countrycolor = 'rgb(204, 204, 204)',
    ),
)


fig.show()

In [ ]:
import json
df =  pd.read_csv('../data/joined_datasets/us-state-boundaries.csv', sep = ";")[['St Asgeojson']]
df.rename({"St Asgeojson": "Coords"}, inplace = True, axis = 1)
def flatten_coords(json_str):
   data = json.loads(json_str)
   return [coord for polygon in data["coordinates"] for ring in polygon for coord in ring]
df["flattened_coords"] = df['Coords'].apply(flatten_coords)

json.loads(df["Coords"][0])